# Week 3 Helper Notebook

> WARNING: This code was generated as a skeleton for you all and may not work right out of the box. It was developed with assistance from Claude and not fully tested on the eICU dataset. Thus, if something doesn't work, or you see something missing, you should engineer it yourself. This is STARTER code and should not be treated as the final code!!

In this notebook you will train a **Variational Autoencoder (VAE)** on a table with both
**categorical** columns (e.g. diagnoses) and **continuous** columns (e.g. heart rate). 

1. First, naviagate to the `tabular_vae` repository. **Follow the installation structions in the `README.md` for setting up pytorch.** We will also be using Weights and Biases to track model performance. You will need to restart the notebook after installing pytorch / wandb. 


2. Then read thru all the files in the `tabular_vae` folder. You do not need to edit them, but do read them!


| File | Purpose |
|---|---|
| `encoders.py` | [only read if you're doing Project 3] turns one continuous column into `m >= 1` numbers (and back). Put in your code from Week 2! |
| `preprocessing.py` | one-hot encodes categoricals, applies the numerical preprocessinig, builds design matrix per patient |
| `vae.py` | the VAE model and the loss function |

This notebook runs on the **CPU**, no GPU (CUDA) needed.

In [ ]:
%load_ext autoreload
%autoreload 2
# This code helps with autoloading new files / modules. 

In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import pickle as pk

from IPython.display import display
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from proj34_representation.tabular_vae.encoders import *
from proj34_representation.tabular_vae.vae import TabularVAE, vae_loss
from proj34_representation.tabular_vae.preprocessing import TabularPreprocessor

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## Step 1: Load eICU data

In [ ]:
feature_sets = pk.load(open('../../data/feature_names.pkl', 'rb'))
patient_agg = pd.read_parquet('../../data/clean_dataset.parquet')

print("Dataset shape:", patient_agg.shape)
patient_agg.head()

In [ ]:
# For privacy, eICU records patients older than 89 using the string "> 89".
# This is how I chose to handle this, but you are free to change it! 
if 'age' in patient_agg.columns:
    patient_agg.loc[patient_agg.age == '> 89', 'age'] = 90
    patient_agg['age'] = patient_agg['age'].astype(float)

In [ ]:
X_cols = #TODO: import final list of X columns from last week 
df = patient_agg[X_cols]

In [ ]:
# Run this as is! 

num_cols = [x for x in X_cols if x in
    feature_sets['labs'].tolist()
    + feature_sets['vitals'].tolist()
    + ['age', 'admissionheight', 'admissionweight']
]

bin_cols = [x for x in X_cols if x in 
    feature_sets['icd10_before24h'].tolist()
    + feature_sets['admissiondx'].tolist()
]

cat_cols = [x for x in X_cols if x in [
    'hospital_region',
    'ethnicity',
    'gender',
    'hospital_numbedscategory',
    'hospitaldischargeyear',
    'hospitalid'
]]
cat_cols = bin_cols + cat_cols

In [ ]:
# TODO: copy over and freeze splits based on Week 2 so the splits should be the exact same every time
# Make sure you have a heldout test split ie use train_test_split from sklearn twice, ie first for trainval / test, and then for train /val 
train_df, val_df, test_df = #train_test_split....

## Step 1: Detour: Missingness

You may have noticed that variables are often missing in our dataset. For example, age is missing for 0.0346% of all patients, the lab measurement of ANF/ANA is missing for 99.9%, and height is missing for 2%.

Categorical variables can be missing too. For example, if a patient's gender was not recorded.

However, missing values are easier to handle in categorical variables than in continuous ones, because when we one-hot encode the variable, we can simply add one more column for "missing". For example:

|Patient| Recorded gender  | female | male | missing |
|:--|------------------|:------:|:----:|:-------:|
|A| female           |   1    |  0   |    0    |
|B| male             |   0    |  1   |    0    |
|C| *(not recorded)* |   0    |  0   |    1    |

The model treats "missing" like any other category, so it can learn how common it is and even generate it in synthetic data.

The challenge with continuous variables is that we need a numerical input to our model, because NaNs are not valid inputs (a single NaN in a row makes the loss NaN, and then all the weights become NaN too).

**Your goal is to briefly explore and pick one way of handling missingness for continuous variables.** Note, how you define missing values will change how you choose to model continuous variables. Some suggestions are:


1. **Drop the patients** that contain ANY missing value. Simple, but you may throw away a lot of data. 
2. **Fill with a constant**, such as the mean or median (computed on the training set only!). Easy, but many patients now share exactly the same value and the data can lose some predictive power. Use `sklearn.impute.SimpleImputer`. 
4. **Fill with the median and add a "was it missing?" binary variable**, the same trick as for categorical variables. Again use median of training data.
    - Note if you do this: You will need to add on these extra "was missing" variable (# new binary cols = # continuous variables) to the list above of `bin_cols`
6. **Predict the missing value from the other columns** on the training data only (for example a regression, k-nearest neighbors, or a MICE-style imputer such as scikit-learn's `IterativeImputer`). If you use imputation, you are free to use AI to help you impute the dataframe above !!
7. **Multiple imputation.** Fill the gaps several different (random) ways to get several complete datasets, then use a different one in each epoch. If you use imputation, you are free to use AI to help you impute the dataframe above !!


For example, say you build some Imputer class. You would want to do 
```
imp = Imputer()
imp.fit(train_df numerical data)

imputed train = imp.transform(train_df numerical data)
imputed val = imp.transform(val_df numerical data)

```


In [ ]:
# TODO:
# first for categorical data, just fill in the missing values with the string Missing. 
# -Ask yourself, why? What might this information preserve? 

imputed_train_df_cat = # TODO
imputed_val_df_cat = # TODO

In [ ]:
# TODO: visualize new imputed values!!

In [ ]:
# TODO! 
# Pick an imputation / missingness strategy for numerical data 
imputed_train_df_num, imputed_val_df_num = #TODO

In [ ]:
imputed_train_df = # TODO joint numerical and categorical and make sure its a dataframe
imputed_val_df = # TODO joint numerical and categorical  and make sure its a dataframe

assert imputed_train_df.isnull().sum().sum() == 0, "Train set has missing values after imputation"
assert imputed_val_df.isnull().sum().sum() == 0, "Val set has missing values after imputation"

## Step 2: Let's train a simple VAE model 

In [ ]:

# ----------------------- Settings you can play with -----------------------
EPOCHS = 100            # how many passes over the training data
BATCH_SIZE = 256        # rows per gradient update
LEARNING_RATE = 1e-3    # step size of the optimizer
HIDDEN_DIM = 64         # width of the hidden layers
LATENT_DIM = 8          # size of z
BETA = 1.0              # weight of the KL term in the loss
KL_WARMUP_EPOCHS = 10   # BETA grows slowly from ~0 to its full value during these epochs
VAL_FRACTION = 0.2      # share of the data held out for validation
SEED = 42

USE_WANDB = False       # set to True to log to Weights & Biases (needs `pip install wandb` and `wandb login`)
WANDB_PROJECT = "tabular-vae"
# --------------------------------------------------------------------------

torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:

# By default every continuous column uses StandardEncoder.
# For Proj3: To use another encoder for a column, pass a dictionary where the value
# is an encoder in the encoders.py, for example:
#     cont_encoders = {"age": ModeSpecificEncoder()} You don't need to do this this week though. 
preprocessor = TabularPreprocessor(cat_cols, num_cols, cont_encoders=None)

x_train = preprocessor.fit_transform(imputed_train_df)      # fit on TRAIN only
val_df = preprocessor.drop_unseen(imputed_val_df)           # (rare) categories never seen in training
x_val = preprocessor.transform(imputed_val_df)

print("Number of categories per categorical column:", preprocessor.cat_dims_)
print("Encoded width of each continuous column:    ", preprocessor.cont_dims_)
print(f"Each row has {preprocessor.input_dim_} numbers "
      f"({preprocessor.cat_total_} categorical + {preprocessor.cont_total_} continuous)")
print(f"Rows: {len(x_train)} train, {len(x_val)} validation")
new_cols = preprocessor.get_feature_names()
# A peek at the first 3 preprocessed rows:
pd.DataFrame(x_train[:3],columns=new_cols).round(2)

In [ ]:
# A `DataLoader` hands the data to the model in small random **batches**.
train_loader = DataLoader(TensorDataset(torch.from_numpy(x_train)), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.from_numpy(x_val)), batch_size=BATCH_SIZE, shuffle=False)

## 3. The model

Open `vae.py` and read `TabularVAE`. It has an **encoder** (row → `mu`, `logvar`), a sampling step
(`z = mu + sigma * eps`), and a **decoder** (`z` → reconstructed row).

In [ ]:
model = TabularVAE(
    input_dim=preprocessor.input_dim_,
    cat_dims=preprocessor.cat_dims_,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print("Number of trainable parameters:", sum(p.numel() for p in model.parameters()))

## 4. The loss function

`vae_loss` (in `vae.py`) adds up **three** terms, each averaged per row:

| Term | Meaning | Formula used |
|---|---|---|
| `recon_cat` | did we reconstruct the categorical variables? | cross-entropy, one per categorical variable |
| `recon_cont` | did we reconstruct the continuous variables? | squared error |
| `kl` | is the latent space close to a standard normal? | KL divergence |

```
loss = recon_cat + recon_cont + beta * kl
```

Below, `run_epoch` does one full pass through a dataset. With an optimizer it **trains**;
without one it only **evaluates** (used for the validation set).

In [ ]:
def run_epoch(model, loader, beta, optimizer=None):
    """One pass over `loader`. 
    Returns the average per-row recon_cat, recon_cont and kl."""
    training = optimizer is not None
    model.train(training)                         # switches between training and evaluation mode

    sums = {"recon_cat": 0.0, "recon_cont": 0.0, "kl": 0.0}
    n_rows = 0

    with torch.set_grad_enabled(training):        # no gradients needed when just evaluating
        # TODO: pass the input into the model and get the loss 
        # hint: look in vae.py. 
        x_hat, mu, logvar = # TODO
        loss, parts = #TODO
        if training:
            optimizer.zero_grad()             # 1) clear old gradients
            loss.backward()                   # 2) compute new gradients
            optimizer.step()                  # 3) update the weights

        n_rows += x.size(0)
        for name in sums:
            sums[name] += parts[name] * x.size(0)
    return {name: value / n_rows for name, value in sums.items()}

## 5. Train

!! IMPORTANT !! DO NOT use test set to tune hyperparemeters!! (Why?)


Every epoch we train on the training set and then measure the loss on the validation set.

Two details:

* **KL warm-up:** at the start, the KL weight is small and grows to `BETA`. This helps the model learn
  to use `z` before the KL term starts pulling it towards N(0, 1).
* The *logged* total loss always uses the full `BETA`, so the numbers are comparable across epochs.
* We keep a copy of the model from the epoch with the **lowest validation loss**.

In [ ]:
if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, config={
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE, "hidden_dim": HIDDEN_DIM,
        "latent_dim": LATENT_DIM, "beta": BETA, "kl_warmup_epochs": KL_WARMUP_EPOCHS,
    })


def full_loss(parts):
    return parts["recon_cat"] + parts["recon_cont"] + BETA * parts["kl"]


history = []                         # one dictionary of numbers per epoch
best_val_loss = float("inf")
best_state = None

for epoch in range(1, EPOCHS + 1):
    beta_now = BETA * min(1.0, epoch / max(1, KL_WARMUP_EPOCHS))      # KL warm-up

    train_parts = run_epoch(model, train_loader, beta_now, optimizer)   # learn
    val_parts = run_epoch(model, val_loader, beta_now)                  # just measure

    row = {"epoch": epoch, "train_loss": full_loss(train_parts), "val_loss": full_loss(val_parts)}
    for name in ["recon_cat", "recon_cont", "kl"]:
        row[f"train_{name}"] = train_parts[name]
        row[f"val_{name}"] = val_parts[name]
    history.append(row)

    if USE_WANDB:
        wandb.log(row)

    note = ""
    if row["val_loss"] < best_val_loss:
        best_val_loss = row["val_loss"]
        best_state = copy.deepcopy(model.state_dict())
        note = "  <- best so far"

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:3d} | train loss {row['train_loss']:7.3f} | val loss {row['val_loss']:7.3f}"
              f" | val: cat {row['val_recon_cat']:5.3f}, cont {row['val_recon_cont']:5.3f}, kl {row['val_kl']:5.3f}{note}")

model.load_state_dict(best_state)    # go back to the best epoch
history = pd.DataFrame(history)
print(f"\nBest validation loss: {best_val_loss:.3f}")

if USE_WANDB:
    wandb.finish()

## 6. Understand the loss curves

* **Train and validation both go down:** the model is learning.
* **Train goes down but validation goes up:** *overfitting* (the model memorizes the training rows).
* **KL close to 0:** *posterior collapse*. The decoder ignores `z`, so generated data will look generic.
  Try a smaller `BETA` or a longer `KL_WARMUP_EPOCHS`.

In [ ]:
panels = [("loss", "Total loss"),
          ("recon_cat", "Categorical reconstruction"),
          ("recon_cont", "Continuous reconstruction"),
          ("kl", "KL divergence")]

fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for ax, (key, title) in zip(axes, panels):
    ax.plot(history["epoch"], history[f"train_{key}"], label="train")
    ax.plot(history["epoch"], history[f"val_{key}"], label="validation")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Repeat until loss goes down. 

In [ ]:
# TODO: keep trying this, debugigng the code, etc. until the loss functions look as they should. 
# For "what should they look like", look at the curves here:
# https://dmol.pub/dl/VAE.html

# If you are having trouble getting your model to work, that's totally ok! 
# "Getting your model to work" means you are now one step closer to being a CS (AI) researcer.
# There are a lot of resources out there (including some I've provided) that can 
# help you figure out what is going on and how to fix it!

In [ ]:
# TODO: try different latent dims. what happens to the different loss terms? 
# [this type of comparison is exactly what weights and biases excels at]
# Hypothesize what increasing and decreasing the latent dims do 

## Exercise: Are the z's any good?

A common way to check a learned representation is a **linear probe**: if a simple linear model can predict an outcome Y from the latent vector z, then z has captured useful information about x.

1. Pick an outcome Y for every sample (one that was not an input to the VAE).
2. Pass X through the trained encoder and keep the **mean vector `mu`** as z (no sampling).
3. Fit a linear model on the training rows only: **z → Y**. Use linear regression for a continuous Y, or logistic regression for a binary one.
4. Evaluate on the validation rows, using the **same train/validation split** as the VAE.
5. Compare with a linear model trained on **x → Y**. Which does better? Why might z be worse (or better)?

*Bonus:* try different `LATENT_DIM` values. How does probe performance change?

In [ ]:
# TODO: recover Y from the same train / val split of the data. Have Y be something 
# like future mortality or future diagnosis. Make sure Y doesn't have missing values
# before you try to predict it! And make sure the indexes of Y match the indexes of X.

In [ ]:
# TODO